# Internet Archive — Feasibility Check for Austrian Advocacy Domains

This notebook checks **two things** for each candidate domain before any scraping decision:

1. **Archive coverage** via the Wayback CDX API — how many snapshots exist, across what years
2. **Live robots.txt** — what the site currently allows automated access to

The CDX API is a metadata API (no content fetched), so this check is lightweight and fast.

### Candidate domains
Austrian NGOs, advocacy groups, and public institutions relevant to *Kindeswohl* / *Entfremdung* discourse.

In [1]:
import requests
import pandas as pd
import time
from collections import Counter
from urllib.robotparser import RobotFileParser
from urllib.error import URLError
import json

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 50)

/Users/maksimsmirnov/Desktop/thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# ── Candidate domains ──────────────────────────────────────────────────────────
# Grouped by type. Add/remove freely as you discover more relevant organizations.
# Format: (domain, short_label, group)

DOMAINS = [
    # Fathers' rights / shared parenting advocacy
    ("vaeteraufbruch.at",            "Väteraufbruch AT",         "fathers_rights"),
    ("vaeter.at",                    "Väter AT",                  "fathers_rights"),
    ("maennerberatung.at",           "Männerberatung",            "fathers_rights"),

    # Child welfare / child protection
    ("kinderschutz.at",              "Kinderschutz AT",           "child_welfare"),
    ("kinderrechte.at",              "Kinderrechte AT",           "child_welfare"),
    ("kinderschutzzentrum.at",       "Kinderschutzzentrum",       "child_welfare"),
    ("neustart.at",                  "Neustart",                  "child_welfare"),

    # Family law / legal aid
    ("familienrecht.at",             "Familienrecht AT",          "family_law"),
    ("ifs.at",                       "ifs Sozialberatung",        "family_law"),
    ("rainbows.at",                  "Rainbows",                  "family_law"),

    # Women / domestic violence context
    ("aoef.at",                      "AOEF",                      "womens_orgs"),
    ("frauenhelpline.at",            "Frauenhelpline",            "womens_orgs"),
    ("interventionsstelle-wien.at",  "Interventionsstelle Wien",  "womens_orgs"),

    # Government / institutional
    ("bmj.gv.at",                    "Justizministerium",         "government"),
    ("help.gv.at",                   "help.gv.at",                "government"),
    ("familie.at",                   "Familie AT (gov)",          "government"),
]

print(f"Total domains to check: {len(DOMAINS)}")
for group in sorted(set(d[2] for d in DOMAINS)):
    members = [d[1] for d in DOMAINS if d[2] == group]
    print(f"  {group}: {', '.join(members)}")

Total domains to check: 16
  child_welfare: Kinderschutz AT, Kinderrechte AT, Kinderschutzzentrum, Neustart
  family_law: Familienrecht AT, ifs Sozialberatung, Rainbows
  fathers_rights: Väteraufbruch AT, Väter AT, Männerberatung
  government: Justizministerium, help.gv.at, Familie AT (gov)
  womens_orgs: AOEF, Frauenhelpline, Interventionsstelle Wien


## Step 1 — Check robots.txt on live sites

We use Python's built-in `RobotFileParser`. We check whether a generic scraper (`*`) is allowed to access the root path.

**Important:** robots.txt is not legally binding under EU TDM law (Article 3, DSM Directive), but respecting it is good practice and protects your research from reputational risk. A `Disallow: /` is a signal to be cautious and potentially reach out to the organization.

In [3]:
def check_robots(domain, user_agent="*", path="/", timeout=8):
    """
    Fetch and parse robots.txt for a domain.
    Returns a dict with: allowed (bool), crawl_delay, raw_url, error
    """
    robots_url = f"https://{domain}/robots.txt"
    rp = RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        allowed = rp.can_fetch(user_agent, f"https://{domain}{path}")
        delay = rp.crawl_delay(user_agent)
        return {"allowed": allowed, "crawl_delay": delay, "robots_url": robots_url, "error": None}
    except Exception as e:
        # Many small NGO sites have no robots.txt — that means unrestricted
        return {"allowed": None, "crawl_delay": None, "robots_url": robots_url, "error": str(e)[:60]}


robots_results = []
for domain, label, group in DOMAINS:
    result = check_robots(domain)
    result["domain"] = domain
    result["label"] = label
    result["group"] = group
    robots_results.append(result)
    status = "✓ allowed" if result["allowed"] is True else ("✗ blocked" if result["allowed"] is False else "? no robots.txt")
    print(f"{label:<30} {status}")
    time.sleep(0.5)  # polite

robots_df = pd.DataFrame(robots_results)[["group", "label", "domain", "allowed", "crawl_delay", "error"]]

Väteraufbruch AT               ? no robots.txt
Väter AT                       ? no robots.txt
Männerberatung                 ? no robots.txt
Kinderschutz AT                ✓ allowed
Kinderrechte AT                ✓ allowed
Kinderschutzzentrum            ✓ allowed
Neustart                       ? no robots.txt
Familienrecht AT               ✓ allowed
ifs Sozialberatung             ✓ allowed
Rainbows                       ✓ allowed
AOEF                           ✓ allowed
Frauenhelpline                 ✓ allowed
Interventionsstelle Wien       ✓ allowed
Justizministerium              ✓ allowed
help.gv.at                     ✓ allowed
Familie AT (gov)               ✓ allowed


In [4]:
# Summary table
print("\n── robots.txt summary ──")
display(robots_df.sort_values(["group", "allowed"], ascending=[True, False]))


── robots.txt summary ──


,group,label,domain,allowed,crawl_delay,error
3,child_welfare,Kinderschutz AT,kinderschutz.at,True,NaN,None
4,child_welfare,Kinderrechte AT,kinderrechte.at,True,NaN,None
5,child_welfare,Kinderschutzzentrum,kinderschutzzentrum.at,True,NaN,None
6,child_welfare,Neustart,neustart.at,None,NaN,<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certific...
7,family_law,Familienrecht AT,familienrecht.at,True,NaN,None
8,family_law,ifs Sozialberatung,ifs.at,True,NaN,None
9,family_law,Rainbows,rainbows.at,True,NaN,None
0,fathers_rights,Väteraufbruch AT,vaeteraufbruch.at,None,NaN,"<urlopen error [Errno 8] nodename nor servname provided,..."
1,fathers_rights,Väter AT,vaeter.at,None,NaN,<urlopen error [Errno 51] Network is unreachable>
2,fathers_rights,Männerberatung,maennerberatung.at,None,NaN,<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certific...


## Step 2 — CDX API: Archive coverage per domain

The [CDX API](https://github.com/internetarchive/wayback/tree/master/wayback-cdx-server) returns metadata about archived snapshots — **no content is fetched**. 

We ask for:
- Total snapshot count (all status codes, all years)
- Year distribution of HTTP 200 snapshots (the ones actually usable)
- First and last snapshot dates

**Reading the results:** You want domains with:
- At least a few dozen snapshots across multiple years
- Coverage reaching back to ~2005–2010 (when the legal discourse around Entfremdung intensified in Austria)
- Consistent coverage, not just one or two years

In [5]:
CDX_URL = "http://web.archive.org/cdx/search/cdx"

def get_cdx_summary(domain, timeout=15):
    """
    Query CDX API for a domain and return coverage summary.
    Uses matchType=domain to catch www. and subdomains.
    """
    base_params = {
        "url": domain,
        "matchType": "domain",
        "output": "json",
        "fl": "timestamp,statuscode",
        "collapse": "timestamp:6",  # one entry per month max — keeps response manageable
        "limit": 2000,
    }

    try:
        r = requests.get(CDX_URL, params=base_params, timeout=timeout)
        r.raise_for_status()
        rows = r.json()

        if not rows or len(rows) <= 1:  # CDX returns header row + data rows
            return {"total_snapshots": 0, "ok_snapshots": 0, "first_year": None,
                    "last_year": None, "years_covered": [], "error": None}

        header = rows[0]  # ['timestamp', 'statuscode']
        data = rows[1:]   # actual snapshot rows

        ts_idx = header.index("timestamp")
        sc_idx = header.index("statuscode")

        total = len(data)
        ok_rows = [r for r in data if r[sc_idx] == "200"]
        years = sorted(set(r[ts_idx][:4] for r in ok_rows))

        return {
            "total_snapshots": total,
            "ok_snapshots": len(ok_rows),
            "first_year": years[0] if years else None,
            "last_year": years[-1] if years else None,
            "years_with_ok": len(years),
            "years_list": years,
            "error": None
        }

    except Exception as e:
        return {"total_snapshots": 0, "ok_snapshots": 0, "first_year": None,
                "last_year": None, "years_with_ok": 0, "years_list": [], "error": str(e)[:80]}


print("Querying CDX API (this may take 1-2 min)...\n")
cdx_results = []
for domain, label, group in DOMAINS:
    result = get_cdx_summary(domain)
    result["domain"] = domain
    result["label"] = label
    result["group"] = group
    cdx_results.append(result)
    print(f"{label:<30} total={result['total_snapshots']:>4}  ok={result['ok_snapshots']:>4}  "
          f"years={result['first_year']}–{result['last_year']}  ({result.get('years_with_ok', 0)} yrs with content)")
    time.sleep(1.0)  # be polite to the Archive

Querying CDX API (this may take 1-2 min)...

Väteraufbruch AT               total=   0  ok=   0  years=None–None  (0 yrs with content)
Väter AT                       total=   0  ok=   0  years=None–None  (0 yrs with content)
Männerberatung                 total=1810  ok=1471  years=2001–2012  (12 yrs with content)
Kinderschutz AT                total=2000  ok=1637  years=2000–2026  (27 yrs with content)
Kinderrechte AT                total=   0  ok=   0  years=None–None  (0 yrs with content)
Kinderschutzzentrum            total=2000  ok=1561  years=2000–2026  (27 yrs with content)
Neustart                       total=2000  ok= 740  years=2002–2026  (25 yrs with content)
Familienrecht AT               total=   0  ok=   0  years=None–None  (0 yrs with content)
ifs Sozialberatung             total=   0  ok=   0  years=None–None  (0 yrs with content)
Rainbows                       total=   0  ok=   0  years=None–None  (0 yrs with content)
AOEF                           total=   0  ok=   0 

In [6]:
# Build a clean summary dataframe
cdx_df = pd.DataFrame([
    {
        "group": r["group"],
        "label": r["label"],
        "domain": r["domain"],
        "total_snapshots": r["total_snapshots"],
        "ok_snapshots": r["ok_snapshots"],
        "first_year": r["first_year"],
        "last_year": r["last_year"],
        "years_with_content": r.get("years_with_ok", 0),
        "error": r.get("error"),
    }
    for r in cdx_results
])

# Sort by usability: ok_snapshots desc
display(cdx_df.sort_values("ok_snapshots", ascending=False))

,group,label,domain,total_snapshots,ok_snapshots,first_year,last_year,years_with_content,error
3,child_welfare,Kinderschutz AT,kinderschutz.at,2000,1637,2000,2026,27,None
5,child_welfare,Kinderschutzzentrum,kinderschutzzentrum.at,2000,1561,2000,2026,27,None
2,fathers_rights,Männerberatung,maennerberatung.at,1810,1471,2001,2012,12,None
11,womens_orgs,Frauenhelpline,frauenhelpline.at,2000,1468,2002,2026,23,None
13,government,Justizministerium,bmj.gv.at,2000,1413,2000,2026,17,None
12,womens_orgs,Interventionsstelle Wien,interventionsstelle-wien.at,2000,1126,2002,2023,19,None
6,child_welfare,Neustart,neustart.at,2000,740,2002,2026,25,None
0,fathers_rights,Väteraufbruch AT,vaeteraufbruch.at,0,0,None,None,0,None
1,fathers_rights,Väter AT,vaeter.at,0,0,None,None,0,"HTTPConnectionPool(host='web.archive.org', port=80): Rea..."
4,child_welfare,Kinderrechte AT,kinderrechte.at,0,0,None,None,0,"HTTPConnectionPool(host='web.archive.org', port=80): Rea..."


## Step 3 — Combined feasibility score

We combine robots.txt and CDX results into a simple feasibility assessment.

**Scoring logic:**
- `ok_snapshots >= 20` and `years_with_content >= 5` → **High** potential
- `ok_snapshots >= 5` and `years_with_content >= 2` → **Medium** potential  
- Otherwise → **Low**

Robots.txt blocking doesn't disqualify (EU TDM law), but it's flagged as a caution.

In [ ]:
# Merge robots + CDX
merged = cdx_df.merge(
    robots_df[["domain", "allowed", "crawl_delay"]],
    on="domain",
    how="left"
)

def feasibility(row):
    ok = row["ok_snapshots"]
    yrs = row["years_with_content"]
    if ok >= 20 and yrs >= 5:
        return "🟢 High"
    elif ok >= 5 and yrs >= 2:
        return "🟡 Medium"
    else:
        return "🔴 Low"

def robots_flag(row):
    if row["allowed"] is False:
        return "⚠ blocked"
    elif row["allowed"] is None:
        return "— no file"
    else:
        return "✓ ok"

merged["feasibility"] = merged.apply(feasibility, axis=1)
merged["robots_status"] = merged.apply(robots_flag, axis=1)

# Sort feasibility best→worst via an explicit categorical, not raw string order
# (raw string sort relies on emoji codepoint accidents).
feasibility_order = ["🟢 High", "🟡 Medium", "🔴 Low"]
merged["feasibility"] = pd.Categorical(
    merged["feasibility"], categories=feasibility_order, ordered=True
)

summary = merged[[
    "feasibility", "group", "label", "ok_snapshots",
    "first_year", "last_year", "years_with_content", "robots_status", "crawl_delay"
]].sort_values(["feasibility", "ok_snapshots"], ascending=[True, False])

print("── Final feasibility summary ──")
display(summary)

In [8]:
# Print just the High + Medium candidates worth pursuing
worthy = summary[summary["feasibility"].isin(["🟢 High", "🟡 Medium"])]
print(f"Domains worth pursuing: {len(worthy)}\n")

for _, row in worthy.iterrows():
    print(f"{row['feasibility']}  {row['label']:<30} "
          f"{row['ok_snapshots']} snapshots  "
          f"{row['first_year']}–{row['last_year']}  "
          f"{row['robots_status']}")

Domains worth pursuing: 7

🟢 High  Kinderschutz AT                1637 snapshots  2000–2026  ✓ ok
🟢 High  Kinderschutzzentrum            1561 snapshots  2000–2026  ✓ ok
🟢 High  Männerberatung                 1471 snapshots  2001–2012  — no file
🟢 High  Frauenhelpline                 1468 snapshots  2002–2026  ✓ ok
🟢 High  Justizministerium              1413 snapshots  2000–2026  ✓ ok
🟢 High  Interventionsstelle Wien       1126 snapshots  2002–2023  ✓ ok
🟢 High  Neustart                       740 snapshots  2002–2026  — no file


In [9]:
# Save results for reference
summary.to_json("../data/wayback_feasibility.json", orient="records", indent=2, force_ascii=False)
print("Saved to data/wayback_feasibility.json")

Saved to data/wayback_feasibility.json


## Step 4 — Preview actual snapshot URLs for a candidate domain

Once you identify a promising domain above, run this cell to see what actual pages are available — so you can assess what *kind* of content the Archive captured (homepage only? deep pages? PDFs?).

In [22]:
# ── Set this to whichever domain looked most promising ──
INSPECT_DOMAIN = "maennerberatung.at"   # ← change me after seeing the results above

params = {
    "url": INSPECT_DOMAIN,
    "matchType": "domain",
    "output": "json",
    "fl": "timestamp,original,statuscode,mimetype",
    "filter": "statuscode:200",
    "limit": 200,
}

# web.archive.org is often slow — retry once on timeout/connection errors before giving up
def fetch_cdx(timeout):
    last_err = None
    for attempt in (1, 2):
        try:
            return requests.get(CDX_URL, params=params, timeout=timeout)
        except (requests.Timeout, requests.ConnectionError) as e:
            last_err = e
            print(f"  attempt {attempt} failed ({type(e).__name__}); retrying..." if attempt == 1 else "")
            time.sleep(2)
    raise last_err

try:
    r = fetch_cdx(timeout=45)
except (requests.Timeout, requests.ConnectionError) as e:
    print(f"CDX API unreachable for {INSPECT_DOMAIN}: {type(e).__name__}")
else:
    if not r.ok:
        print(f"CDX API error {r.status_code} for {INSPECT_DOMAIN}")
    elif not r.text.strip():
        print(f"No snapshots found in the Wayback Machine for: {INSPECT_DOMAIN}")
    else:
        rows = r.json()
        if len(rows) <= 1:
            print(f"No 200-OK snapshots for {INSPECT_DOMAIN} (CDX returned header only)")
        else:
            header, data = rows[0], rows[1:]

            snap_df = pd.DataFrame(data, columns=header)
            snap_df["year"] = snap_df["timestamp"].str[:4]
            snap_df["wayback_url"] = "https://web.archive.org/web/" + snap_df["timestamp"] + "/" + snap_df["original"]

            print(f"Domain: {INSPECT_DOMAIN}")
            print(f"Total 200-OK snapshots sampled: {len(snap_df)}")
            print(f"\nSnapshots per year:")
            print(snap_df.groupby("year").size().to_string())
            print(f"\nMIME types captured:")
            print(snap_df["mimetype"].value_counts().head(10).to_string())
            print(f"\nSample URLs (first 10):")
            for url in snap_df["original"].head(10):
                print(" ", url)

Domain: maennerberatung.at
Total 200-OK snapshots sampled: 200

Snapshots per year:
year
2001    10
2002    17
2003    26
2004    27
2005    19
2006     6
2007    10
2008     5
2009    15
2010    25
2011    24
2012    16

MIME types captured:
mimetype
text/html    200

Sample URLs (first 10):
  http://www.maennerberatung.at:80/
  http://www.maennerberatung.at:80/
  http://www.maennerberatung.at:80/
  http://www.maennerberatung.at:80/
  http://www.maennerberatung.at:80/
  http://www.maennerberatung.at:80/
  http://www.maennerberatung.at:80/
  http://www.maennerberatung.at:80/
  http://www.maennerberatung.at:80/
  http://www.maennerberatung.at:80/


In [23]:
# What sub-paths were captured? Useful to see if the Archive got beyond the homepage.
from urllib.parse import urlparse

paths = snap_df["original"].apply(lambda u: urlparse(u).path)
depth = paths.apply(lambda p: len([x for x in p.split("/") if x]))

print("Path depth distribution (0 = homepage only, higher = deeper content):")
print(depth.value_counts().sort_index().to_string())

print("\nMost-archived sub-paths:")
print(paths.value_counts().head(20).to_string())

Path depth distribution (0 = homepage only, higher = deeper content):
original
0    80
1    63
2    57

Most-archived sub-paths:
original
/                                                        80
/angebote_gewalttaetigkeit.html                          23
/angebote_einsamkeit.html                                21
/angebote_beziehungskrisen.html                          19
/0002_emailberatung/0002_emailberatung.html               6
/0003_kontakt/0003_kontakt.html                           6
/0001_inDerKrise/0001_inDerKrise.html                     6
/0007_haftungsausschluss/0007_haftungsausschluss.html     6
/0006_englishVersion/0006_englishVersion.html             6
/0004_wirUeberUns/00042_poelzler.html                     5
/0100_persoenlicheBeratung/0102_gewalt.html               5
/0100_persoenlicheBeratung/0109_mannUndRecht.html         5
/0004_wirUeberUns/00042_mitarbeiterInnen.html             4
/0004_wirUeberUns/00042_reinbacher.html                   3
/0004_wirUeberUns/0004

## What to do with these results

**High/Medium feasibility + deep path coverage** → good candidate for content scraping. Build a targeted scraper that:
- Fetches only HTML snapshots (filter mimetype=text/html)
- Samples one snapshot per year (not every monthly capture)
- Extracts body text + page title + year
- Looks for keyword presence: `Entfremdung`, `Kindeswohl`, `Sorgerecht`, `Umgang`

**High feasibility + homepage-only coverage** → longitudinal framing analysis of mission statements / landing page language only. Still interesting but limited.

**Low feasibility** → skip or manually check one or two snapshots on web.archive.org to see if it's worth a one-off manual read.

**Next step:** If you find 3–5 high-quality domains, a separate scraping notebook can fetch the actual content and feed it into your NLP pipeline alongside the RIS Rechtssätze.